# Resistencia antimicrobiana (AMR) — Unión, EDA y limpieza (2000–2024)

En este notebook unificamos los CSV de **resistencia antimicrobiana** por **bacteria**, **grupo antibiótico**, **país** y **año**.

**Objetivos:**
1. Cargar y concatenar únicamente los CSV de **resistencia** desde `../data/raw`
2. Mantener columnas relevantes y renombrarlas
3. Ejecutar EDA estructurado para validar calidad
4. Aplicar limpieza mínima (valores faltantes y tipos)
5. Exportar el dataset final a `../data/processed`

In [44]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

In [45]:
RAW_DIR = Path("../data/raw")

# Detectar CSV de resistencia
files = [
    f for f in RAW_DIR.glob("*.csv")
    if "-" in f.name
]

# Cargar y concatenar
df_raw_amr = pd.concat(
    (pd.read_csv(file) for file in files),
    ignore_index=True
)

### Inspección inicial del dataset raw

Antes de aplicar cualquier transformación, se realiza una exploración preliminar del dataset concatenado.  

El objetivo es comprender su estructura, tipos de datos y posibles inconsistencias, con el fin de identificar problemas que deban corregirse en la fase de limpieza.

In [46]:
print("===== PRIMERA INSPECCIÓN DEL DATASET RAW =====\n")

print("Dimensiones:", df_raw_amr.shape)
print("\nColumnas:")
display(df_raw_amr.columns)

print("\nPrimeras filas:")
display(df_raw_amr.head())

print("\nÚltimas filas:")
display(df_raw_amr.tail())

print("\nMuestra aleatoria:")
display(df_raw_amr.sample(5))

print("\nTipos de datos:")
display(df_raw_amr.dtypes)

===== PRIMERA INSPECCIÓN DEL DATASET RAW =====

Dimensiones: (6161, 9)

Columnas:


Index(['HealthTopic', 'Population', 'Indicator', 'Unit', 'Time', 'RegionCode',
       'RegionName', 'NumValue', 'TxtValue'],
      dtype='object')


Primeras filas:


,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
0,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,BG,Bulgaria,58.461538460,NaN
1,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,CY,Cyprus,52.173913040,NaN
2,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,DE,Germany,5.882352940,NaN
3,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,DK,Denmark,10.389610380,NaN
4,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,EL,Greece,78.119935170,NaN



Últimas filas:


,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
6156,Antimicrobial resistance,Klebsiella pneumoniae|Fluoroquinolones,"R - resistant isolates, percentage",%,2024,PT,Portugal,39.566982400,NaN
6157,Antimicrobial resistance,Klebsiella pneumoniae|Fluoroquinolones,"R - resistant isolates, percentage",%,2024,RO,Romania,64.513350550,NaN
6158,Antimicrobial resistance,Klebsiella pneumoniae|Fluoroquinolones,"R - resistant isolates, percentage",%,2024,SE,Sweden,13.551779930,NaN
6159,Antimicrobial resistance,Klebsiella pneumoniae|Fluoroquinolones,"R - resistant isolates, percentage",%,2024,SI,Slovenia,20.738636360,NaN
6160,Antimicrobial resistance,Klebsiella pneumoniae|Fluoroquinolones,"R - resistant isolates, percentage",%,2024,SK,Slovakia,50.131926120,NaN



Muestra aleatoria:


,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
2792,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2012,UK,United Kingdom,13.137912760,NaN
2798,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2013,DE,Germany,10.740393620,NaN
6027,Antimicrobial resistance,Klebsiella pneumoniae|Fluoroquinolones,"R - resistant isolates, percentage",%,2020,HR,Croatia,54.104477610,NaN
2313,Antimicrobial resistance,Escherichia coli|Carbapenems,"R - resistant isolates, percentage",%,2020,EE,Estonia,0.000000000,NaN
3045,Antimicrobial resistance,Escherichia coli|Third-generation cephalosporins,"R - resistant isolates, percentage",%,2021,HU,Hungary,20.404858290,NaN



Tipos de datos:


HealthTopic     object
Population      object
Indicator       object
Unit            object
Time             int64
RegionCode      object
RegionName      object
NumValue        object
TxtValue       float64
dtype: object

### Tratamiento de valores "-" en NumValue

Durante la inspección inicial se detecta que la columna `NumValue` utiliza el carácter `"-"` para representar datos faltantes.  

Este valor no es interpretado como `NaN` por pandas, lo que impide identificar correctamente los valores nulos y realizar conversiones numéricas adecuadas.

Por ello, se reemplaza `"-"` por `NaN` antes de continuar con el proceso de limpieza.

In [47]:
df_raw_amr["NumValue"] = (
    df_raw_amr["NumValue"]
    .replace("-", np.nan)
)

### Análisis exploratorio del dataset raw

Una vez corregida la representación de valores faltantes (`"-"` → `NaN`), se realiza un análisis exploratorio completo del dataset original.  

El objetivo es comprender su estructura, distribución de variables, presencia de valores nulos, rango temporal y posibles duplicados antes de aplicar transformaciones adicionales.

Este paso permite validar el estado real de los datos y justificar las decisiones posteriores de limpieza y modelado.

In [48]:
def eda_amr(df):
    """
    Analisis exploratorio del dataset AMR (version raw post-concat).
    Se ejecuta tras corregir los "-" como NaN.
    """

    print("===== EDA DATASET RAW AMR =====\n")

    # 1️⃣ Vista general
    print("Dimensiones (filas, columnas):", df.shape)
    print("\nColumnas:")
    display(df.columns)

    print("\nPrimeras filas:")
    display(df.head())

    print("\nTipos de datos:")
    display(df.dtypes)

    # 2️⃣ Estadisticos descriptivos
    print("\nESTADISTICAS DESCRIPTIVAS (numericas):")
    display(df.describe().T)

    print("\nESTADISTICAS DESCRIPTIVAS (categoricas):")
    display(df.describe(include="object").T)


    # 3️⃣ Cardinalidad
    print("\nCARDINALIDAD (valores unicos por columna):")
    display(df.nunique())


    # 4️⃣ Rango temporal
    print("\nRANGO TEMPORAL:")
    print("Min year:", df["Time"].min())
    print("Max year:", df["Time"].max())


    # 5️⃣ Valores nulos
    print("\nVALORES NULOS (conteo):")
    display(df.isnull().sum())

    print("\nVALORES NULOS (%):")
    display(((df.isnull().sum() / df.shape[0]) * 100).round(2))


    # 6️⃣ Duplicados estructurales (clave logica en raw)
    print("\nDUPLICADOS (Population + Time + RegionCode):")
    dup_count = df.duplicated(
        subset=["Population", "Time", "RegionCode"]
    ).sum()

    print("Numero de duplicados:", dup_count)


    # 7️⃣ Distribucion resistencia
    print("\nDISTRIBUCION NumValue (resistencia):")
    display(df["NumValue"].describe())

In [49]:
eda_amr(df_raw_amr)

===== EDA DATASET RAW AMR =====

Dimensiones (filas, columnas): (6161, 9)

Columnas:


Index(['HealthTopic', 'Population', 'Indicator', 'Unit', 'Time', 'RegionCode',
       'RegionName', 'NumValue', 'TxtValue'],
      dtype='object')


Primeras filas:


,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
0,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,BG,Bulgaria,58.461538460,NaN
1,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,CY,Cyprus,52.173913040,NaN
2,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,DE,Germany,5.882352940,NaN
3,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,DK,Denmark,10.389610380,NaN
4,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,EL,Greece,78.119935170,NaN



Tipos de datos:


...

HealthTopic       0
Population        0
Indicator         0
Unit              0
Time              0
RegionCode        0
RegionName        0
NumValue        262
TxtValue       6161
dtype: int64


VALORES NULOS (%):


HealthTopic      0.00
Population       0.00
Indicator        0.00
Unit             0.00
Time             0.00
RegionCode       0.00
RegionName       0.00
NumValue         4.25
TxtValue       100.00
dtype: float64


DUPLICADOS (Population + Time + RegionCode):
Numero de duplicados: 0

DISTRIBUCION NumValue (resistencia):


count            5899
unique           4754
top       0.000000000
freq              563
Name: NumValue, dtype: object

### Conclusiones del análisis exploratorio

El análisis exploratorio confirma que el dataset cubre el periodo 2000–2024 y presenta una estructura consistente a nivel de país, año y combinación bacteria–antibiótico.

Tras la corrección de los valores "-" como `NaN`, los valores faltantes quedan correctamente identificados y pueden ser tratados adecuadamente en las fases posteriores.

No se detectan duplicados estructurales en la combinación clave (Population, Time, RegionCode), por lo que no es necesario aplicar eliminación adicional de registros.

Con esta validación, el dataset está listo para su estandarización y transformación estructural.

## Selección y estandarización de columnas relevantes

El dataset original contiene 9 columnas.  
Para el análisis nos quedamos únicamente con las necesarias:

- Population → información de bacteria + grupo antibiótico
- Time → año
- RegionCode → código país
- RegionName → país
- NumValue → porcentaje de resistencia

Renombramos las columnas para facilitar su uso posterior.

In [50]:
# Selección y renombrado de columnas relevantes
df_amr = (
    df_raw_amr[
        ["Population", "Time", "RegionCode", "RegionName", "NumValue"]
    ]
    .rename(columns={
        "Population": "bacteria_antibiotic",
        "Time": "year",
        "RegionCode": "country_code",
        "RegionName": "country",
        "NumValue": "resistance_percentage"
    })
)

# Reordenar columnas
df_amr = df_amr[
    ["year", "country_code", "country", "bacteria_antibiotic", "resistance_percentage"]
]

## Separación de bacteria y antibiótico

La columna `bacteria_antibiotic` contiene información combinada en el formato:

Bacteria | Grupo antibiótico

Para facilitar el análisis y permitir filtros independientes en Tableau,
separamos esta columna en:

- `bacteria`
- `antibiotic_group`

Posteriormente eliminamos la columna original.

In [51]:
# Separar bacteria y grupo antibiótico
df_amr[["bacteria", "antibiotic_group"]] = (
    df_amr["bacteria_antibiotic"].str.split("|", expand=True)
)

# Eliminar columna original y reordenar
df_amr = df_amr.drop(columns="bacteria_antibiotic")[
    ["year", "country_code", "country",
     "bacteria", "antibiotic_group", "resistance_percentage"]
]

### Limpieza básica del dataset de resistencia

En esta fase se aplican transformaciones mínimas para preparar el dataset para análisis posteriores:

- Conversión de `resistance_percentage` a tipo numérico.
- Limpieza de espacios en columnas de texto.

El objetivo es garantizar consistencia estructural antes de continuar con la integración con el dataset de consumo.

In [52]:
def clean_amr_dataset(df):
    """
    Limpieza básica del dataset AMR:
    - Conversión de tipos
    - Limpieza de texto
    - Eliminación de duplicados estructurales
    """

    df = df.copy()

    # Conversión de tipos
    df["resistance_percentage"] = df["resistance_percentage"].astype(float)

    # Limpieza de texto
    text_cols = ["country", "bacteria", "antibiotic_group"]
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()

    return df


# Ejecutar limpieza
df_amr_clean = clean_amr_dataset(df_amr)

## Creación de columna ATC común para integración con consumo

Para poder analizar conjuntamente consumo y resistencia,
creamos una nueva columna `atc_group` que mapea los grupos de resistencia
a los grupos ATC utilizados en el dataset de consumo.

Reglas de mapeo:

- Aminoglycosides → J01G
- Carbapenems → J01D
- Third-generation cephalosporins → J01D
- Fluoroquinolones → J01M

Esta columna permitirá relacionar ambos datasets directamente.

In [53]:
# Diccionario de mapeo AMR → ATC
atc_mapping = {
    "Aminoglycosides": "J01G",
    "Carbapenems": "J01D",
    "Third-generation cephalosporins": "J01D",
    "Fluoroquinolones": "J01M"
}

# Aplicar mapeo
df_amr_clean["atc_group"] = df_amr_clean["antibiotic_group"].map(atc_mapping)

## Exportación final del dataset de resistencia

Se exporta la versión limpia y alineada (2000–2024),
incluyendo la columna `atc_group` que permite integrar
directamente con el dataset de consumo.

Este archivo constituye la versión oficial procesada del dataset AMR.

In [54]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "antimicrobial_resistance_2000_2024_processed.csv"

df_amr_clean.to_csv(output_path, index=False)